# Gruppen-CV: Sind die Scores bei unbekannten Aufnahmen belastbar?

**Autor:** jd · **Datum:** 2026-10-20

**Ziel:** Prüfen, ob der zufällige Split zu optimistisch ist (#170). Zeilen derselben Zelle aus AEZ × Monat stammen sehr wahrscheinlich aus derselben Hyperion-Aufnahme, also mit derselben Atmosphäre und Beleuchtung. Zufällige Folds verteilen eine Aufnahme auf Training und Test. Das Modell kann dann die Aufnahme wiedererkennen, statt die Pflanze.

**Hypothese:** Mit Gruppen-Folds fällt die CV-BAcc deutlich, am stärksten wenn ganze Monate fehlen. Das Stadium hängt am Monat und leidet dann mehr als die Kultur.

**Vorgehen:** Vier Eingaben werden mit vier Fold-Arten je neu getunt (gleiches Raster wie in `33_jd_datenreduktion`). Bewertet wird nur auf den Trainingsfolds. Der Validierungsteil ist selbst zufällig gezogen und hilft bei dieser Frage nicht.

| Fold-Art | Was im Test-Fold fehlt im Training | Prüft |
| --- | --- | --- |
| `random` | nichts (geteilte Folds aus `make data`) | bisherige Referenz |
| `aez_month` | ganze Aufnahmen (AEZ × Monat) | Übertragung auf eine neue Aufnahme |
| `month` | ganze Monate | Übertragung auf einen neuen Zeitpunkt |
| `aez` | ganze Regionen | Übertragung auf eine neue Region |

Alle Läufe stehen im MLflow-Experiment `split-leakage`.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

from awp2.config import (
    MLFLOW_SPLIT_EXPERIMENT,
    MODEL_DOCS_FIGURES_DIR,
    MONTH_COL,
    REDUCTION_BEST_PARAMS_COL,
    REDUCTION_VARIANT_COL,
)
from awp2.data import load_split
from awp2.evaluation import METRIC_LABELS, evaluate
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions
from awp2.models.combined import make_combined_svm
from awp2.plots import save_doc_figure
from awp2.preprocessing import PreprocessingConfig
from awp2.reduction import CURVE_METRICS, ReductionVariant, reduction_curve, reduction_variants

## 1. Eingaben und Fold-Arten

- **Alle Bänder** und **Sentinel-2:** die Referenz und die Empfehlung aus #155.
- **Nur Spektren:** ohne AEZ und Monat. Zeigt, wie gut das Spektrum allein übertragbar ist.
- **Nur Meta:** ohne Spektrum. Zeigt, wie stark der Kontext an bekannte Aufnahmen gebunden ist.

Die Familien-Spalte aus `reduction_curve` hat hier keine Bedeutung.

In [ ]:
APPROACH = "combined"
GROUPINGS = ("random", "aez_month", "month", "aez")
model = make_combined_svm()
base = PreprocessingConfig(use_spectral_standard_scale=True)
grid = {
    "estimator__C": [30.0, 100.0, 300.0],
    "estimator__gamma": [0.001, 0.003, 0.01, 0.03, "scale"],
}
standard = {v.name: v for v in reduction_variants(base)}
variants = [
    standard["all_bands"],
    standard["sentinel2"],
    ReductionVariant(
        name="no_meta",
        family="all_bands",
        preprocessing=base.model_copy(update={"use_meta": False}),
    ),
    standard["meta_only"],
]
[v.name for v in variants]

## 2. Vergleich der Fold-Arten

Jede Kombination aus Fold-Art und Eingabe wird neu getunt. Danach wird die beste Einstellung pro Fold für Kultur, Stadium und kombiniert bewertet.

In [ ]:
curves = []
for grouping in GROUPINGS:
    curve = reduction_curve(
        model,
        variants,
        TuneConfig(
            name=f"split_{grouping}",
            approach=APPROACH,
            experiment=MLFLOW_SPLIT_EXPERIMENT,
            description=f"Split-Leakage-Check: Folds {grouping}",
            param_grid=grid,
            preprocessing=base,
            folds=grouping,
        ),
    )
    curves.append(curve.assign(folds=grouping))
results = pd.concat(curves, ignore_index=True)
results.pivot(index=REDUCTION_VARIANT_COL, columns="folds", values="bacc_combined")[
    list(GROUPINGS)
].round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, metric in zip(axes, CURVE_METRICS, strict=True):
    means = results.pivot(index="folds", columns=REDUCTION_VARIANT_COL, values=metric)
    stds = results.pivot(index="folds", columns=REDUCTION_VARIANT_COL, values=f"{metric}_std")
    order = [v.name for v in variants]
    means.loc[list(GROUPINGS), order].plot.bar(
        ax=ax, yerr=stds.loc[list(GROUPINGS), order], rot=0, capsize=2, width=0.8
    )
    ax.set_title(f"{METRIC_LABELS[metric]} (CV)")
    ax.set_xlabel("Fold-Art")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize="small", loc="lower left")
fig.suptitle("Zufällige vs. Gruppen-Folds")
fig.tight_layout()
save_doc_figure(fig, "gruppen_cv_vergleich", MODEL_DOCS_FIGURES_DIR)

**Befund:**

- **Der zufällige Split ist für unbekannte Aufnahmen stark optimistisch.** Die CV-BAcc kombiniert fällt von 0,848 auf 0,38 (neue Aufnahme), 0,42 (neuer Monat) und 0,30 (neue Region). Kultur fällt von 0,92 auf etwa 0,52–0,66, Stadium von 0,89 auf etwa 0,51–0,59.
- **Die Hypothese der gemeinsamen Aufnahme stützt sich vor allem auf die Fold-Art `aez_month`:**
    - Dort sieht das Training denselben Monat in anderen Regionen und dieselbe Region in anderen Monaten. Trotzdem halbiert sich die Qualität.
    - Auch das reine Spektrum (`no_meta`) fällt von 0,777 auf 0,33. Das Modell lernt also nicht nur den Kontext auswendig. Auch die Spektren tragen aufnahmetypische Eigenheiten wie Atmosphäre, Beleuchtung und den Stand der Saison an diesem Tag.
- **Neuer Monat:** Das Stadium verliert mehr als die Kultur (0,51 gegenüber 0,66). Teils lässt sich das gar nicht lernen: Manche Paare kommen nur in einem Monat vor, z. B. `cotton|Harvest`.
- **Neue Region:** Hier verliert die Kultur am meisten (0,52). Die Streuung ist groß (± 0,11–0,15), weil die 7 AEZ sehr unterschiedlich groß sind.
- **Die Datenreduktion bleibt gültig:** Sentinel-2 liegt bei jeder Fold-Art gleichauf mit allen Bändern (0,398 / 0,426 / 0,287 gegenüber 0,383 / 0,421 / 0,299). Die Empfehlung aus #155 hängt also nicht am Split.

## 3. Wo bricht es ein? Aufschlüsselung pro Monat

Für `month`-Folds werden die Out-of-fold-Vorhersagen der Referenz (alle Bänder, beste Einstellung) pro Monat bewertet. Jeder Monat wurde dabei von einem Modell vorhergesagt, das diesen Monat nie gesehen hat.

In [ ]:
reference = results[(results["folds"] == "month") & (results[REDUCTION_VARIANT_COL] == "all_bands")]
best_params = reference[REDUCTION_BEST_PARAMS_COL].iloc[0]
oof = out_of_fold_predictions(
    make_combined_svm().set_params(**best_params),
    RunConfig(
        name="split_month_oof", approach=APPROACH, preprocessing=base, folds="month", track=False
    ),
)
split = load_split()
by_month = pd.DataFrame(
    {
        month: evaluate(split.y_train.loc[rows.index], oof.loc[rows.index]).model_dump()
        for month, rows in split.X_train.groupby(MONTH_COL)
    }
).T[list(CURVE_METRICS)]
by_month.round(3)

In [ ]:
month_ax = by_month.rename(columns=METRIC_LABELS).plot.bar(rot=0, figsize=(9, 4.5))
month_ax.set_xlabel("Monat (im Training nie gesehen)")
month_ax.set_ylabel("BAcc (out-of-fold)")
month_ax.set_title("Referenz-SVM mit Monats-Folds: Qualität pro Monat")
month_ax.grid(axis="y", alpha=0.3)
save_doc_figure(month_ax.figure, "gruppen_cv_monate", MODEL_DOCS_FIGURES_DIR)

**Befund pro Monat:**

- **Kein Monat ist sicher, wenn er im Training fehlt.** Die BAcc kombiniert liegt in jedem Monat zwischen 0,35 und 0,47.
- **Kultur:** Juli wird am besten erkannt (0,75), mitten in der Saison und bei voller Biomasse. August am schlechtesten (0,58).
- **Stadium:** Juli ist umgekehrt am schwächsten (0,39). Im Juli liegen „Early_Mid“, „Late“ und „Mature_Senesc“ nebeneinander, und ohne Juli-Beispiele fehlt die Abgrenzung.

## 4. Welche Schätzung gilt für die Abgabe?

Entscheidend ist, aus welchen Aufnahmen der Testdatensatz der Challenge stammt.

In [ ]:
from awp2.config import AEZ_COL
from awp2.data import load_test, load_train

train_cells = set(zip(load_train()[AEZ_COL], load_train()[MONTH_COL], strict=True))
test = load_test()
test_cells = set(zip(test[AEZ_COL], test[MONTH_COL], strict=True))
{
    "Zellen im Test": len(test_cells),
    "davon nicht im Training": len(test_cells - train_cells),
    "Testzeilen": len(test),
}

## Fazit

| Fold-Art | Alle Bänder | Sentinel-2 | Nur Spektren | Nur Meta |
| --- | --- | --- | --- | --- |
| zufällig | 0,848 | 0,852 | 0,777 | 0,537 |
| neue Aufnahme (AEZ × Monat) | 0,383 | 0,398 | 0,326 | 0,291 |
| neuer Monat | 0,421 | 0,426 | 0,340 | 0,279 |
| neue Region (AEZ) | 0,299 | 0,287 | 0,250 | 0,282 |

*CV-BAcc kombiniert, Tuning-Runs im Experiment `split-leakage`.*

- **Für die Challenge bleibt der zufällige Split die richtige Schätzung.** Der Testdatensatz (1.397 Zeilen) stammt aus denselben 32 Zellen aus AEZ × Monat wie das Training, keine Zelle ist neu. Unsere 0,85–0,87 beschreiben genau diese Situation.
- **Die Modelle sind an bekannte Aufnahmen gebunden.** Für eine neue Saison, eine neue Region oder auch nur eine neue Aufnahme wären etwa 0,3–0,4 BAcc kombiniert realistisch. Das gehört als zentrale Grenze in die Modellkarte und den Abschlussbericht.
- **Was das für die übrigen Arbeiten heißt:**
    - Die Empfehlung Sentinel-2 aus #155 gilt bei jeder Fold-Art.
    - Merkmale, die direkt aus AEZ × Monat gelernt werden (Phänologie-Prior #172), helfen nur innerhalb bekannter Zellen. Sie verstärken die Bindung an bekannte Aufnahmen.
    - Echte Übertragbarkeit bräuchte Korrekturen pro Aufnahme, z. B. eine Normalisierung je Aufnahme oder Merkmale, die gegen Unterschiede der Atmosphäre robust sind. Das ist außerhalb des Projektumfangs, aber ein guter Ausblick für den Bericht.